1.Data Ingestion

In [1]:
from langchain.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
import os
file_path = os.path.join(os.getcwd(), "data", "sample.pdf")

In [3]:
loader = PyPDFLoader(file_path)

In [4]:
document = loader.load()


In [5]:
len(document)

77

2. spilt data into the chunks

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

In [ ]:
chunks = text_splitter.split_documents(document)
chunks

In [8]:
len(chunks)

615

In [9]:
chunks[0].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2023-07-20T00:30:36+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2023-07-20T00:30:36+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': 'c:\\Users\\Asus\\OneDrive\\Desktop\\TanmayFiles\\DocumentPortal\\DocumentPortal\\notebook\\data\\sample.pdf',
 'total_pages': 77,
 'page': 0,
 'page_label': '1'}

In [10]:
chunks[1].page_content

'Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich\nYinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra\nIgor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi\nAlan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang\nRoss Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang\nAngela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic'

3. Embed and store

In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")


In [12]:
from langchain.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding_model)

In [13]:
vector_store

1. In memory -faiss 
2. On disk
3. on cloud

#This is the retriever process 
means from the vectorstore we get most relevant document most appropriate k result

In [19]:
vs = vector_store.similarity_search("What is llm model?",k=10)

In [ ]:
vs

In [58]:
retriver = vector_store.as_retriever(search_kwargs={"k": 10})

creating the RAG chain

In [36]:
from langchain_groq import ChatGroq

llm=ChatGroq(model="deepseek-r1-distill-llama-70b")

In [28]:
from langchain_core.prompts import PromptTemplate

In [59]:
prompt_template ="""
Answer the question based on the context provided below.
If the context does not provide suffucient information respond with:
"I don not have enough information to answer that question."

Context: {context}

Question: {question}

Answer:


"""

In [60]:
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"])


In [61]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [62]:
from langchain_core.runnables import RunnablePassthrough

In [63]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [64]:
rag_chain =(
    {"context":retriver | format_docs, "question":RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

In [65]:

result = rag_chain.invoke("can you tell me Scaling trends for the reward model?")

In [66]:
result

"<think>\nOkay, I need to answer the question about scaling trends for the reward model based on the provided context. Let me go through the context step by step to gather the necessary information.\n\nFirst, the context mentions that they studied scaling trends in terms of data and model size for the reward model. They fine-tuned different model sizes using an increasing amount of reward model data collected each week. There's a reference to Table 26 for details on volume per batch, and Figure 6 shows these trends.\n\nFrom Figure 6, it's noted that larger models achieve higher performance with a similar volume of data, which is expected. Additionally, the scaling performance is mentioned in the context of Helpfulness and Safety reward models, which were used to determine the best settings.\n\nThe context also talks about experiments where larger models perform better on more distinct response pairs but may regress on similar samples. This suggests that while scaling up helps with more